In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Working folder:", Path.cwd())

In [ ]:
#Generate the synthetic source extracts.
from pathlib import Path
import csv
import random
from datetime import date, timedelta

rng = random.Random(3602026)
root = Path('data')
raw = root / 'raw'
raw.mkdir(parents=True, exist_ok=True)
for folder in ['quarantine', 'curated']:
    (root / folder).mkdir(parents=True, exist_ok=True)

def write_csv(name, rows):
    path = raw / name
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f'{path}: {len(rows):,} rows')

cities = ['Lagos', 'Abuja', 'Ibadan', 'Port Harcourt']
properties = [dict(property_id=f'P{i:03}', property_name=f'MetroSpace Tower {i:02}',
                   city=cities[(i-1) % len(cities)]) for i in range(1, 13)]
units = []
for p in properties:
    for j in range(1, 21):
        units.append(dict(unit_id=f"{p['property_id']}-U{j:03}", property_id=p['property_id'],
                          floor_area_sqm=rng.choice([45, 60, 75, 90, 120, 150]), active=1))
leases = []
for i, unit in enumerate(units, start=1):
    if rng.random() < 0.22:  # intentionally vacant unit
        continue
    start = date(2025, rng.randint(1, 12), rng.randint(1, 25))
    end = date(2026, rng.randint(3, 12), rng.randint(1, 28)) if rng.random() < 0.27 else None
    leases.append(dict(lease_id=f'L{i:05}', unit_id=unit['unit_id'],
                       start_date=start.isoformat(), end_date=end.isoformat() if end else '',
                       monthly_rent_usd=rng.choice([650, 800, 975, 1200, 1500, 1800, 2200])))
priorities = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
work_orders = []
for i in range(1, 1501):
    unit = rng.choice(units)
    opened = date(2026, rng.randint(1, 6), rng.randint(1, 28))
    resolved = rng.random() < 0.82
    response = round(rng.uniform(1, 96), 1)
    work_orders.append(dict(work_order_id=f'WO{i:06}', unit_id=unit['unit_id'],
        opened_date=opened.isoformat(), closed_date=(opened + timedelta(days=rng.randint(0, 15))).isoformat() if resolved else '',
        priority=rng.choice(priorities), response_hours=response if resolved else '',
        cost_usd=round(rng.uniform(20, 1250), 2), status='RESOLVED' if resolved else 'OPEN'))
months = [f'2026-{m:02}-01' for m in range(1, 7)]
readings = []
for unit in units:
    for month in months:
        readings.append(dict(reading_id=f'R{len(readings)+1:06}', unit_id=unit['unit_id'],
                             reading_month=month, energy_kwh=round(rng.uniform(100, 1250), 2)))
# Known intentionally dirty records; do not remove these from the raw source.
work_orders.append(work_orders[8].copy())                     # exact duplicate
work_orders[34]['unit_id'] = 'P999-U999'                       # unknown parent
work_orders[52]['response_hours'] = ''                         # resolved with missing response
work_orders[63]['cost_usd'] = '-300'                           # invalid negative repair cost
readings[17]['energy_kwh'] = '-45'                             # invalid negative consumption
readings[34]['unit_id'] = 'P999-U999'                           # unknown unit
units.append(units[4].copy())                                 # duplicate identifier
for name, rows in [('properties.csv', properties), ('units.csv', units), ('leases.csv', leases),
                   ('work_orders.csv', work_orders), ('meter_readings.csv', readings)]:
    write_csv(name, rows)
print('Raw fixture ready. Retain a copy before any transformations.')

In [ ]:
from pathlib import Path
import pandas as pd

raw_dir = Path("data/raw")

for path in sorted(raw_dir.glob("*.csv")):
    df = pd.read_csv(path)
    print(f"{path.name}: {len(df):,} rows, {len(df.columns)} columns")

In [1]:
import pandas as pd

# Import our reusable ingestion functions and source definitions.
from src.ingest import (
    RAW_DIR,
    EXPECTED_COLUMNS,
    ingest_source,
    ingest_all,
)

# Load the five datasets and save their audit manifest.
datasets, ingestion_manifest = ingest_all()

# Display the audit records in the notebook as submission evidence.
display(pd.DataFrame(ingestion_manifest))

,filename,row_count,ingest_timestamp_utc,sha256
0,properties.csv,12,2026-10-16T20:04:00.117129+00:00,c9865bcf2b142026c5dd05547f9c14d3e11f5a27a3e22b...
1,units.csv,241,2026-10-16T20:04:00.121640+00:00,f9bcef3d08d4662c270bfff385299306995ed6ba3f0c54...
2,leases.csv,198,2026-10-16T20:04:00.125646+00:00,b6a8e10a11dfbca4358d5a9d60ad4709bc6511417137ae...
3,work_orders.csv,1501,2026-10-16T20:04:00.141648+00:00,835c05c0410f7a481d71f4efa8213520d012688ebb8b75...
4,meter_readings.csv,1440,2026-10-16T20:04:00.151647+00:00,93684a8ed99f3e4f368646d2f766a41805d54e7db5ef43...


In [2]:
from pathlib import Path
# Test 1: Confirm that a missing file raises the expected exception.
try:
    ingest_source(
        RAW_DIR / "nonexistent.csv",
        EXPECTED_COLUMNS["properties.csv"]
    )
except FileNotFoundError as exc:
    # This is the expected outcome, so the test passes.
    print("PASS — missing-file check:", exc)
else:
    # Fail the test if ingestion unexpectedly succeeds.
    raise AssertionError("Expected FileNotFoundError")


# Test 2: Reload each source to check that ingestion is repeatable.
for original_entry in ingestion_manifest:
    filename = original_entry["filename"]

    repeated_df, repeated_entry = ingest_source(
        RAW_DIR / filename,
        EXPECTED_COLUMNS[filename]
    )

    # Matching hashes confirm that the source bytes have not changed.
    assert repeated_entry["sha256"] == original_entry["sha256"], (
        f"Raw file changed: {filename}"
    )

    # Compare values, column order and data types between both loads.
    pd.testing.assert_frame_equal(
        repeated_df,
        datasets[Path(filename).stem]
    )

# Timestamps may differ, but source hashes and loaded data must match.
print("PASS — repeated ingestion returns identical data and raw hashes")

PASS — missing-file check: Source file not found: C:\Users\Cedri\OneDrive\Documents\10Alytics - Data Engineering\Final Capstone Project\data\raw\nonexistent.csv
PASS — repeated ingestion returns identical data and raw hashes


In [3]:
from tempfile import TemporaryDirectory
from pathlib import Path

# Use temporary files so the actual raw data remains unchanged.
with TemporaryDirectory() as temporary_folder:
    test_dir = Path(temporary_folder)

    # Each case contains invalid CSV content and its expected error message.
    test_cases = [
        (
            "missing_columns.csv",
            "property_id\nP001\n",
            "missing required columns"
        ),
        (
            "empty.csv",
            "",
            "Source file is empty"
        ),
        (
            "header_only.csv",
            "property_id,property_name,city\n",
            "contains no data rows"
        ),
    ]

    for filename, content, expected_message in test_cases:
        test_path = test_dir / filename
        test_path.write_text(content, encoding="utf-8")

        try:
            ingest_source(
                test_path,
                EXPECTED_COLUMNS["properties.csv"]
            )
        except ValueError as exc:
            # Ensure the failure happened for the intended reason.
            assert expected_message in str(exc), str(exc)
            print(f"PASS — {filename}: {exc}")
        else:
            raise AssertionError(
                f"Expected ValueError for {filename}"
            )

# Temporary test files are automatically deleted when the block ends.

PASS — missing_columns.csv: missing_columns.csv is missing required columns: city, property_name
PASS — empty.csv: Source file is empty: empty.csv
PASS — header_only.csv: Source file contains no data rows: header_only.csv


### Data Profiling

In [4]:
# Import reusable quality functions before profiling and validation.
import numpy as np
import pandas as pd
from uuid import uuid4

from src.quality import (
    IDENTIFIERS,
    NUMERIC_RULES,
    profile_table,
    separate_duplicates,
    separate_invalid_foreign_keys,
    validate_numeric_fields,
    validate_dates_and_categories,
)

# Start one quality run; reuse this ID throughout validation and saving.
run_id = uuid4().hex


In [5]:
# Build a detailed profile for each of the five datasets.
profiles = {
    name: profile_table(df, IDENTIFIERS[name])
    for name, df in datasets.items()
}

# Display a compact comparison of row counts and duplicates.
profile_summary = pd.DataFrame([
    {
        "dataset": name,
        "row_count": profile["row_count"],
        "duplicate_ids": profile["duplicate_identifier_count"],
        "exact_duplicates": profile["exact_duplicate_count"],
        "blank_fields": sum(profile["blanks_by_column"].values()),
        "null_fields": sum(profile["nulls_by_column"].values()),
    }
    for name, profile in profiles.items()
])

display(profile_summary)

# Show which columns contain blanks or actual nulls.
for name, profile in profiles.items():
    print(f"\n{name}")

    display(pd.DataFrame({
        "data_type": profile["types"],
        "null_count": profile["nulls_by_column"],
        "blank_count": profile["blanks_by_column"],
    }))


,dataset,row_count,duplicate_ids,exact_duplicates,blank_fields,null_fields
0,properties,12,0,0,0,0
1,units,241,1,1,0,0
2,leases,198,0,0,139,0
3,work_orders,1501,1,1,590,0
4,meter_readings,1440,0,0,0,0



properties


,data_type,null_count,blank_count
property_id,string,0,0
property_name,string,0,0
city,string,0,0



units


,data_type,null_count,blank_count
unit_id,string,0,0
property_id,string,0,0
floor_area_sqm,string,0,0
active,string,0,0


...

,data_type,null_count,blank_count
lease_id,string,0,0
unit_id,string,0,0
start_date,string,0,0
end_date,string,0,139
monthly_rent_usd,string,0,0



work_orders


,data_type,null_count,blank_count
work_order_id,string,0,0
unit_id,string,0,0
opened_date,string,0,0
closed_date,string,0,295
priority,string,0,0
response_hours,string,0,295
cost_usd,string,0,0
status,string,0,0



meter_readings


,data_type,null_count,blank_count
reading_id,string,0,0
unit_id,string,0,0
reading_month,string,0,0
energy_kwh,string,0,0


### Inspect missing maintenance fields by status without changing the source data.


In [6]:
# Inspect missing maintenance fields by status without changing the source data.
work_orders = datasets["work_orders"]

# Identify blank fields, allowing for whitespace.
blank_closed = work_orders["closed_date"].str.strip().eq("")
blank_response = work_orders["response_hours"].str.strip().eq("")

# Create a separate diagnostic table.
status_checks = work_orders[["status"]].copy()
status_checks["blank_closed_date"] = blank_closed
status_checks["blank_response_hours"] = blank_response

# Compare missing fields across OPEN and RESOLVED work orders.
display(
    status_checks.groupby("status").agg(
        row_count=("status", "size"),
        missing_closed_date=("blank_closed_date", "sum"),
        missing_response_hours=("blank_response_hours", "sum"),
    )
)

# A resolved order with no response duration needs an explicit SLA policy.
resolved_missing_response = (
    work_orders["status"].eq("RESOLVED")
    & blank_response
)

print(
    "Resolved orders missing response duration:",
    int(resolved_missing_response.sum())
)

display(work_orders.loc[resolved_missing_response])

,row_count,missing_closed_date,missing_response_hours
status,,,
OPEN,295,295,295
RESOLVED,1206,0,0


Resolved orders missing response duration: 0


,work_order_id,unit_id,opened_date,closed_date,priority,response_hours,cost_usd,status


### # Inspect defects without modifying the original DataFrames.


In [7]:
# Inspect defects without modifying the original DataFrames.

units = datasets["units"]
properties = datasets["properties"]
work_orders = datasets["work_orders"]
readings = datasets["meter_readings"]

# Build reference sets for checking parent identifiers.
valid_property_ids = set(properties["property_id"])
valid_unit_ids = set(units["unit_id"])

# Flag units that refer to a property absent from properties.csv.
unknown_property = ~units["property_id"].isin(valid_property_ids)

# Flag work orders and readings that refer to an unknown unit.
unknown_work_order_unit = ~work_orders["unit_id"].isin(valid_unit_ids)
unknown_reading_unit = ~readings["unit_id"].isin(valid_unit_ids)

# Convert values temporarily for inspection.
# Invalid numeric text becomes NaN; the source columns remain unchanged.
repair_cost = pd.to_numeric(work_orders["cost_usd"], errors="coerce")
energy = pd.to_numeric(readings["energy_kwh"], errors="coerce")

# Display records that violate these rules.
checks = {
    "Units with unknown properties": units.loc[unknown_property],
    "Work orders with unknown units": work_orders.loc[
        unknown_work_order_unit
    ],
    "Readings with unknown units": readings.loc[unknown_reading_unit],
    "Negative repair costs": work_orders.loc[repair_cost.lt(0)],
    "Negative energy readings": readings.loc[energy.lt(0)],
}

for description, records in checks.items():
    print(f"\n{description}: {len(records)}")
    display(records)


Units with unknown properties: 0


,unit_id,property_id,floor_area_sqm,active



Work orders with unknown units: 1


,work_order_id,unit_id,opened_date,closed_date,priority,response_hours,cost_usd,status
34,WO000035,P999-U999,2026-04-08,,LOW,,82.75,OPEN



Readings with unknown units: 1


,reading_id,unit_id,reading_month,energy_kwh
34,R000035,P999-U999,2026-05-01,1205.75



Negative repair costs: 1


,work_order_id,unit_id,opened_date,closed_date,priority,response_hours,cost_usd,status
63,WO000064,P003-U019,2026-06-05,2026-06-12,MEDIUM,8.1,-300,RESOLVED



Negative energy readings: 1


,reading_id,unit_id,reading_month,energy_kwh
17,R000018,P001-U003,2026-06-01,-45


In [8]:
# Store intermediate results separately from the raw datasets.
deduplicated_datasets = {}
duplicate_quarantine = {}
duplicate_summary = []

for name, source_df in datasets.items():
    accepted, rejected = separate_duplicates(
        source_df,
        IDENTIFIERS[name],
        run_id,
    )

    deduplicated_datasets[name] = accepted
    duplicate_quarantine[name] = rejected

    duplicate_summary.append({
        "dataset": name,
        "input_rows": len(source_df),
        "accepted_after_duplicate_checks": len(accepted),
        "quarantined_duplicate_rows": len(rejected),
    })

display(pd.DataFrame(duplicate_summary))

,dataset,input_rows,accepted_after_duplicate_checks,quarantined_duplicate_rows
0,properties,12,12,0
1,units,241,240,1
2,leases,198,198,0
3,work_orders,1501,1500,1
4,meter_readings,1440,1440,0


In [9]:
# Copy the duplicate-checked tables for this validation stage.
fk_checked_datasets = {
    name: df.copy()
    for name, df in deduplicated_datasets.items()
}

# Create an empty quarantine table for each dataset.
# Reuse the column structure, including reason_code and run_id.
fk_quarantine = {
    name: rejected.iloc[:0].copy()
    for name, rejected in duplicate_quarantine.items()
}

# Validate units against the duplicate-checked properties.
valid_property_ids = set(
    fk_checked_datasets["properties"]["property_id"]
)

accepted_units, rejected_units = separate_invalid_foreign_keys(
    df=fk_checked_datasets["units"],
    foreign_key="property_id",
    valid_parent_ids=valid_property_ids,
    reason_code="UNKNOWN_PROPERTY",
    run_id=run_id,
)

fk_checked_datasets["units"] = accepted_units
fk_quarantine["units"] = rejected_units

# Use only accepted units as parents for the remaining tables.
valid_unit_ids = set(accepted_units["unit_id"])

for name in ["leases", "work_orders", "meter_readings"]:
    accepted, rejected = separate_invalid_foreign_keys(
        df=fk_checked_datasets[name],
        foreign_key="unit_id",
        valid_parent_ids=valid_unit_ids,
        reason_code="UNKNOWN_UNIT",
        run_id=run_id,
    )

    fk_checked_datasets[name] = accepted
    fk_quarantine[name] = rejected

# Display the results of this stage only.
fk_summary = pd.DataFrame([
    {
        "dataset": name,
        "input_after_duplicate_checks": len(
            deduplicated_datasets[name]
        ),
        "accepted_after_fk_checks": len(fk_checked_datasets[name]),
        "quarantined_fk_rows": len(fk_quarantine[name]),
    }
    for name in fk_checked_datasets
])

display(fk_summary)

,dataset,input_after_duplicate_checks,accepted_after_fk_checks,quarantined_fk_rows
0,properties,12,12,0
1,units,240,240,0
2,leases,198,198,0
3,work_orders,1500,1499,1
4,meter_readings,1440,1439,1


In [10]:
numeric_checked_datasets = {}
numeric_quarantine = {}

for name, source_df in fk_checked_datasets.items():
    accepted, rejected = validate_numeric_fields(
        source_df,
        NUMERIC_RULES[name],
        run_id,
    )

    numeric_checked_datasets[name] = accepted
    numeric_quarantine[name] = rejected

# Show the counts for this validation stage.
display(pd.DataFrame([
    {
        "dataset": name,
        "input_after_fk_checks": len(fk_checked_datasets[name]),
        "accepted_after_numeric_checks": len(
            numeric_checked_datasets[name]
        ),
        "quarantined_numeric_rows": len(numeric_quarantine[name]),
    }
    for name in numeric_checked_datasets
]))

,dataset,input_after_fk_checks,accepted_after_numeric_checks,quarantined_numeric_rows
0,properties,12,12,0
1,units,240,240,0
2,leases,198,198,0
3,work_orders,1499,1498,1
4,meter_readings,1439,1438,1


In [11]:
date_checked_datasets = {}
date_quarantine = {}

for name, source_df in numeric_checked_datasets.items():
    accepted, rejected = validate_dates_and_categories(
        source_df,
        name,
        run_id,
    )

    date_checked_datasets[name] = accepted
    date_quarantine[name] = rejected

display(pd.DataFrame([
    {
        "dataset": name,
        "input_after_numeric_checks": len(
            numeric_checked_datasets[name]
        ),
        "accepted_after_date_checks": len(
            date_checked_datasets[name]
        ),
        "quarantined_date_category_rows": len(
            date_quarantine[name]
        ),
    }
    for name in date_checked_datasets
]))

,dataset,input_after_numeric_checks,accepted_after_date_checks,quarantined_date_category_rows
0,properties,12,12,0
1,units,240,240,0
2,leases,198,198,0
3,work_orders,1498,1498,0
4,meter_readings,1438,1438,0


In [12]:
# Copy the validated tables for the next stage.
business_checked_datasets = {
    name: df.copy()
    for name, df in date_checked_datasets.items()
}

# ---------------------------------------------------------
# 1. Detect multiple readings for the same unit and month.
# ---------------------------------------------------------
readings = business_checked_datasets["meter_readings"]

# Different reading IDs can still refer to the same business key.
# Quarantine all ambiguous records rather than choose one arbitrarily.
duplicate_reading_mask = readings.duplicated(
    subset=["unit_id", "reading_month"],
    keep=False,
)

reading_key_quarantine = readings.loc[
    duplicate_reading_mask
].copy()

reading_key_quarantine["reason_code"] = (
    "DUPLICATE_UNIT_MONTH_READING"
)
reading_key_quarantine["run_id"] = run_id

business_checked_datasets["meter_readings"] = (
    readings.loc[~duplicate_reading_mask]
    .copy()
    .reset_index(drop=True)
)

assert len(readings) == (
    len(business_checked_datasets["meter_readings"])
    + len(reading_key_quarantine)
)

# ---------------------------------------------------------
# 2. Identify overlapping leases within each unit.
# ---------------------------------------------------------
leases = business_checked_datasets["leases"].reset_index(drop=True)

# Retain leases, but flag every lease involved in an overlap.
leases["lease_overlap"] = False
overlap_pairs = []

for unit_id, group in leases.groupby("unit_id"):
    # Check leases in start-date order within the current unit.
    ordered = group.sort_values("start_date")
    indices = ordered.index.to_list()

    for position, left_index in enumerate(indices):
        left = leases.loc[left_index]

        for right_index in indices[position + 1:]:
            right = leases.loc[right_index]

            # Dates are inclusive: a lease starting on another lease's
            # end date is simultaneously active on that day.
            # A missing end date means the lease continues indefinitely.
            overlaps = (
                pd.isna(left["end_date"])
                or right["start_date"] <= left["end_date"]
            )

            if overlaps:
                leases.loc[
                    [left_index, right_index],
                    "lease_overlap"
                ] = True

                overlap_pairs.append({
                    "unit_id": unit_id,
                    "lease_id_1": left["lease_id"],
                    "lease_id_2": right["lease_id"],
                    "reason_code": "OVERLAPPING_LEASES",
                    "run_id": run_id,
                })
            else:
                # Later leases start even later, so no further pair
                # can overlap this left-hand lease.
                break

business_checked_datasets["leases"] = leases

# Keep a separate report explaining each conflicting pair.
lease_overlap_report = pd.DataFrame(
    overlap_pairs,
    columns=[
        "unit_id",
        "lease_id_1",
        "lease_id_2",
        "reason_code",
        "run_id",
    ],
)

print(
    "Quarantined unit-month readings:",
    len(reading_key_quarantine),
)
print("Overlapping lease pairs:", len(lease_overlap_report))
print(
    "Leases flagged for overlap:",
    int(leases["lease_overlap"].sum()),
)

display(lease_overlap_report)

Quarantined unit-month readings: 0
Overlapping lease pairs: 0
Leases flagged for overlap: 0


,unit_id,lease_id_1,lease_id_2,reason_code,run_id


### Test the validation functions

In [13]:
# These tests use separate copies or small synthetic tables.
# They do not modify the project datasets.

# ---------------------------------------------------------
# Test 1: An exact duplicate is quarantined.
# ---------------------------------------------------------
sample = datasets["units"].iloc[[0]].copy()
sample = pd.concat([sample, sample], ignore_index=True)

accepted, rejected = separate_duplicates(
    sample, "unit_id", "test-run"
)

assert len(accepted) == 1
assert len(rejected) == 1
assert rejected.iloc[0]["reason_code"] == "EXACT_DUPLICATE"
print("PASS — exact duplicate quarantined")


# ---------------------------------------------------------
# Test 2: Conflicting records with the same ID are all rejected.
# ---------------------------------------------------------
sample = datasets["units"].iloc[[0]].copy()
conflicting_row = sample.copy()
conflicting_row["floor_area_sqm"] = "999"

sample = pd.concat(
    [sample, conflicting_row],
    ignore_index=True,
)

accepted, rejected = separate_duplicates(
    sample, "unit_id", "test-run"
)

assert accepted.empty
assert len(rejected) == 2
assert rejected["reason_code"].eq("CONFLICTING_IDENTIFIER").all()
print("PASS — conflicting identifiers quarantined")


# ---------------------------------------------------------
# Test 3: An unknown unit is quarantined.
# ---------------------------------------------------------
sample = datasets["work_orders"].iloc[[0]].copy()
sample["unit_id"] = "UNKNOWN-UNIT"

accepted, rejected = separate_invalid_foreign_keys(
    sample,
    "unit_id",
    set(business_checked_datasets["units"]["unit_id"]),
    "UNKNOWN_UNIT",
    "test-run",
)

assert accepted.empty
assert rejected["reason_code"].eq("UNKNOWN_UNIT").all()
print("PASS — unknown foreign key quarantined")


# ---------------------------------------------------------
# Test 4: Negative energy is rejected, while zero is allowed.
# ---------------------------------------------------------
sample = datasets["meter_readings"].iloc[:2].copy()
sample["energy_kwh"] = ["-45", "0"]

accepted, rejected = validate_numeric_fields(
    sample,
    NUMERIC_RULES["meter_readings"],
    "test-run",
)

assert len(accepted) == 1
assert accepted.iloc[0]["energy_kwh"] == 0
assert len(rejected) == 1
assert "NEGATIVE_ENERGY_KWH" in rejected.iloc[0]["reason_code"]
print("PASS — negative energy rejected and zero retained")


# ---------------------------------------------------------
# Test 5: A resolved order missing its response is retained
# but excluded from the SLA-eligible denominator.
# ---------------------------------------------------------
resolved_orders = date_checked_datasets["work_orders"].loc[
    date_checked_datasets["work_orders"]["status"].eq("RESOLVED")
]

sample = resolved_orders.iloc[[0]].copy()
sample["response_hours"] = pd.NA

# Restore source-style date text before testing date parsing.
for column in ["opened_date", "closed_date"]:
    sample[column] = sample[column].dt.strftime("%Y-%m-%d")

accepted, rejected = validate_dates_and_categories(
    sample, "work_orders", "test-run"
)

assert len(accepted) == 1
assert rejected.empty
assert bool(accepted.iloc[0]["missing_sla_response"])
assert not bool(accepted.iloc[0]["sla_eligible"])
print("PASS — missing resolved response flagged and SLA-excluded")


# ---------------------------------------------------------
# Test 6: An impossible calendar date is quarantined.
# ---------------------------------------------------------
sample = datasets["leases"].iloc[[0]].copy()
sample["start_date"] = "2026-02-30"

accepted, rejected = validate_dates_and_categories(
    sample, "leases", "test-run"
)

assert accepted.empty
assert "INVALID_START_DATE" in rejected.iloc[0]["reason_code"]
print("PASS — impossible date quarantined")


# ---------------------------------------------------------
# Test 7: Final parent-child relationships remain valid.
# This catches parents removed by later validation stages.
# ---------------------------------------------------------
valid_properties = set(
    business_checked_datasets["properties"]["property_id"]
)
valid_units = set(
    business_checked_datasets["units"]["unit_id"]
)

assert business_checked_datasets["units"]["property_id"].isin(
    valid_properties
).all()

for name in ["leases", "work_orders", "meter_readings"]:
    assert business_checked_datasets[name]["unit_id"].isin(
        valid_units
    ).all(), f"Invalid final foreign key in {name}"

print("PASS — final foreign keys remain valid")

PASS — exact duplicate quarantined
PASS — conflicting identifiers quarantined
PASS — unknown foreign key quarantined
PASS — negative energy rejected and zero retained
PASS — missing resolved response flagged and SLA-excluded
PASS — impossible date quarantined
PASS — final foreign keys remain valid


### Save the clean staging data, quarantine records

In [14]:
from pathlib import Path
import json
import pandas as pd

# Keep each run's outputs together so earlier evidence is preserved.
output_root = Path("data")
staging_dir = output_root / "staging" / run_id
quarantine_dir = output_root / "quarantine" / run_id
audit_dir = output_root / "curated" / "quality_audit" / run_id

for folder in [staging_dir, quarantine_dir, audit_dir]:
    folder.mkdir(parents=True, exist_ok=True)


def restore_original_values(name, rejected):
    """Restore source text for records rejected after duplicate handling."""
    source = datasets[name]
    identifier = IDENTIFIERS[name]
    source_columns = source.columns.tolist()

    if rejected.empty:
        return pd.DataFrame(
            columns=source_columns + ["reason_code", "run_id"]
        )

    # Conflicting identifiers were already removed during duplicate checks.
    # Remaining identifiers therefore map to one distinct original record.
    original_lookup = source.drop_duplicates().set_index(identifier)

    assert original_lookup.index.is_unique or (
        not original_lookup.index[
            original_lookup.index.duplicated(keep=False)
        ].isin(rejected[identifier]).any()
    ), f"Ambiguous source lookup for {name}"

    # Select only IDs involved in this rejection stage.
    original_rows = source.loc[
        source[identifier].isin(rejected[identifier])
    ].drop_duplicates()

    # Each rejected row must map to exactly one original source record.
    restored = rejected[
        [identifier, "reason_code", "run_id"]
    ].merge(
        original_rows,
        on=identifier,
        how="left",
        validate="many_to_one",
        indicator=True,
    )

    assert restored["_merge"].eq("both").all()
    assert len(restored) == len(rejected)

    return restored[source_columns + ["reason_code", "run_id"]]


clean_datasets = {
    name: df.copy()
    for name, df in business_checked_datasets.items()
}

quarantine_datasets = {}
summary_rows = []

for name, clean_df in clean_datasets.items():
    # Duplicate quarantine already contains original source values.
    rejection_parts = [duplicate_quarantine[name].copy()]

    # Restore original text for later validation stages.
    for stage in [
        fk_quarantine,
        numeric_quarantine,
        date_quarantine,
    ]:
        rejection_parts.append(
            restore_original_values(name, stage[name])
        )

    if name == "meter_readings":
        rejection_parts.append(
            restore_original_values(name, reading_key_quarantine)
        )

    rejected = pd.concat(rejection_parts, ignore_index=True)
    quarantine_datasets[name] = rejected

    input_count = len(datasets[name])
    accepted_count = len(clean_df)
    quarantine_count = len(rejected)

    # Exact duplicates are INCLUDED in the quarantined total.
    duplicate_count = len(duplicate_quarantine[name])

    # Every source row must be accepted or quarantined exactly once.
    assert input_count == accepted_count + quarantine_count, (
        f"Reconciliation failed for {name}"
    )

    # Save typed clean data as CSV for the next stage.
    # Spark will read these using explicitly defined schemas.
    clean_df.to_csv(
        staging_dir / f"{name}.csv",
        index=False,
        date_format="%Y-%m-%d",
    )

    # Save a quarantine CSV even when it contains only column headers.
    rejected.to_csv(
        quarantine_dir / f"{name}.csv",
        index=False,
    )

    summary_rows.append({
        "dataset": name,
        "input_rows": input_count,
        "accepted_rows": accepted_count,
        "quarantined_rows": quarantine_count,
        "duplicate_rows_in_quarantine": duplicate_count,
        "reconciled": input_count == accepted_count + quarantine_count,
        "run_id": run_id,
    })

quality_summary = pd.DataFrame(summary_rows)
quality_summary.to_csv(audit_dir / "reconciliation.csv", index=False)

# Save the detailed raw-data profiles and overlap report.
(audit_dir / "profiles.json").write_text(
    json.dumps(profiles, indent=2),
    encoding="utf-8",
)

lease_overlap_report.to_csv(
    audit_dir / "lease_overlaps.csv",
    index=False,
)

# Record the policies needed to interpret the outputs.
run_details = {
    "run_id": run_id,
    "staging_directory": str(staging_dir),
    "quarantine_directory": str(quarantine_dir),
    "exact_duplicates_counted_as_quarantined": True,
    "missing_resolved_response_policy": (
        "Retain and flag; exclude from SLA-eligible denominator."
    ),
    "overlapping_lease_policy": (
        "Retain and flag; resolve month-end conflicts before KPI aggregation."
    ),
}

(audit_dir / "run_details.json").write_text(
    json.dumps(run_details, indent=2),
    encoding="utf-8",
)

display(quality_summary)
print(f"Clean staging files: {staging_dir}")
print(f"Quarantine files: {quarantine_dir}")
print(f"Quality audit: {audit_dir}")

,dataset,input_rows,accepted_rows,quarantined_rows,duplicate_rows_in_quarantine,reconciled,run_id
0,properties,12,12,0,0,True,3467ad4c2ae2455ca5a25d33deaeb067
1,units,241,240,1,1,True,3467ad4c2ae2455ca5a25d33deaeb067
2,leases,198,198,0,0,True,3467ad4c2ae2455ca5a25d33deaeb067
3,work_orders,1501,1498,3,1,True,3467ad4c2ae2455ca5a25d33deaeb067
4,meter_readings,1440,1438,2,0,True,3467ad4c2ae2455ca5a25d33deaeb067


Clean staging files: data\staging\3467ad4c2ae2455ca5a25d33deaeb067
Quarantine files: data\quarantine\3467ad4c2ae2455ca5a25d33deaeb067
Quality audit: data\curated\quality_audit\3467ad4c2ae2455ca5a25d33deaeb067


### Check PySpark Installation/Configuration

In [15]:
import os
import shutil
import subprocess
from importlib import metadata

# Check whether PySpark is installed in the selected notebook kernel.
try:
    print("PySpark:", metadata.version("pyspark"))
except metadata.PackageNotFoundError:
    print("PySpark: NOT INSTALLED")

# Show any configured Java installation.
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

# Check whether Java is available through PATH.
java_path = shutil.which("java")
print("Java executable:", java_path or "NOT FOUND")

if java_path:
    result = subprocess.run(
        [java_path, "-version"],
        capture_output=True,
        text=True,
    )

    # Java commonly prints version information to stderr.
    print(result.stdout)
    print(result.stderr)

PySpark: 4.0.2
JAVA_HOME: NOT SET
Java executable: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE

java version "21.0.12.1" 2026-08-18 LTS
Java(TM) SE Runtime Environment (build 21.0.12.1+1-LTS-4)
Java HotSpot(TM) 64-Bit Server VM (build 21.0.12.1+1-LTS-4, mixed mode, sharing)



### Installing PySpark in the notebook

In [16]:
%pip install pyspark==4.0.2 pyarrow

     ---------------------------------------- 0.0/434.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/434.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/434.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/434.2 MB ? eta -:--:--
     ---------------------------------------- 0.3/434.2 MB ? eta -:--:--
     ---------------------------------------- 0.3/434.2 MB ? eta -:--:--
     ---------------------------------------- 0.3/434.2 MB ? eta -:--:--
     -------------------------------------- 0.8/434.2 MB 745.8 kB/s eta 0:09:42
     -------------------------------------- 1.0/434.2 MB 968.5 kB/s eta 0:07:28
     -------------------------------------- 1.3/434.2 MB 919.0 kB/s eta 0:07:52
     -------------------------------------- 1.3/434.2 MB 919.0 kB/s eta 0:07:52
     -------------------------------------- 1.6/434.2 MB 953.2 kB/s eta 0:07:34
     -------------------------------------- 1.6/434.2 MB 953.2 kB/s eta 0:07:34
     ----

### Spark Startup

In [16]:
import os
import sys

# Make Spark workers use the same Python environment as this notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# Use two local processing threads for this small project.
spark = (
    SparkSession.builder
    .appName("PropTech360-Capstone")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Reduce routine logging after startup.
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

# Verify that Spark can execute a basic distributed operation.
spark.range(5).show()

Spark version: 4.0.2
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



### Load your validated staging CSVs with explicit schemas so Spark knows which columns are dates, numbers, and Boolean flags.

In [19]:
from pathlib import Path
from pyspark.sql import types as T

# Use the staging directory created by this notebook's quality run.
# Avoid loading dirty files from data/raw.
staging_path = Path(staging_dir).resolve()

if not staging_path.is_dir():
    raise FileNotFoundError(
        f"Staging directory not found: {staging_path}"
    )

# Column order must match the CSV headers written by pandas.
# Numeric fields were saved as decimal values, except the active flag.
SCHEMAS = {
    "properties": T.StructType([
        T.StructField("property_id", T.StringType()),
        T.StructField("property_name", T.StringType()),
        T.StructField("city", T.StringType()),
    ]),

    "units": T.StructType([
        T.StructField("unit_id", T.StringType()),
        T.StructField("property_id", T.StringType()),
        T.StructField("floor_area_sqm", T.DoubleType()),
        T.StructField("active", T.IntegerType()),
    ]),

    "leases": T.StructType([
        T.StructField("lease_id", T.StringType()),
        T.StructField("unit_id", T.StringType()),
        T.StructField("start_date", T.DateType()),
        T.StructField("end_date", T.DateType()),
        T.StructField("monthly_rent_usd", T.DecimalType(18, 2)),
        T.StructField("lease_overlap", T.BooleanType()),
    ]),

    "work_orders": T.StructType([
        T.StructField("work_order_id", T.StringType()),
        T.StructField("unit_id", T.StringType()),
        T.StructField("opened_date", T.DateType()),
        T.StructField("closed_date", T.DateType()),
        T.StructField("priority", T.StringType()),
        T.StructField("response_hours", T.DoubleType()),
        T.StructField("cost_usd", T.DecimalType(18, 2)),
        T.StructField("status", T.StringType()),
        T.StructField("missing_sla_response", T.BooleanType()),
        T.StructField("sla_eligible", T.BooleanType()),
    ]),

    "meter_readings": T.StructType([
        T.StructField("reading_id", T.StringType()),
        T.StructField("unit_id", T.StringType()),
        T.StructField("reading_month", T.DateType()),
        T.StructField("energy_kwh", T.DecimalType(18, 2)),
    ]),
}

spark_datasets = {}

for name, schema in SCHEMAS.items():
    csv_path = staging_path / f"{name}.csv"

    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Staging file not found: {csv_path}"
        )

    # Explicit types avoid schema inference.
    # FAILFAST rejects malformed records.
    # enforceSchema=false checks headers against the supplied schema.
    df = (
        spark.read
        .schema(schema)
        .option("header", True)
        .option("enforceSchema", False)
        .option("mode", "FAILFAST")
        .option("dateFormat", "yyyy-MM-dd")
        .csv(csv_path.as_posix())
    )

    # Cache the full DataFrame and count rows using Spark's DataFrame API.
    df = df.cache()
    row_count = df.count()

    # Read a small sample across all columns to inspect parsed values.
    df.show(2, truncate=False)

    # Compare Spark's loaded count with the validated pandas table.
    expected_count = len(clean_datasets[name])
    assert row_count == expected_count, (
        f"{name}: expected {expected_count}, got {row_count}"
    )

    spark_datasets[name] = df
    print(f"{name}: {row_count:,} rows loaded")

# Inspect an example schema and a small sample.
spark_datasets["leases"].printSchema()
spark_datasets["leases"].show(5, truncate=False)

+-----------+-------------------+-----+
|property_id|property_name      |city |
+-----------+-------------------+-----+
|P001       |MetroSpace Tower 01|Lagos|
|P002       |MetroSpace Tower 02|Abuja|
+-----------+-------------------+-----+
only showing top 2 rows
properties: 12 rows loaded
+---------+-----------+--------------+------+
|unit_id  |property_id|floor_area_sqm|active|
+---------+-----------+--------------+------+
|P001-U001|P001       |75.0          |1     |
|P001-U002|P001       |90.0          |1     |
+---------+-----------+--------------+------+
only showing top 2 rows
units: 240 rows loaded
+--------+---------+----------+--------+----------------+-------------+
|lease_id|unit_id  |start_date|end_date|monthly_rent_usd|lease_overlap|
+--------+---------+----------+--------+----------------+-------------+
|L00001  |P001-U001|2025-03-07|NULL    |975.00          |false        |
|L00003  |P001-U003|2025-08-17|NULL    |975.00          |false        |
+--------+---------+------

In [20]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# 1. Create the January–June 2026 reporting calendar.
# ---------------------------------------------------------
# Build the calendar inside Spark without Python worker operations.
calendar = (
    spark.range(6)
    .select(
        F.add_months(
            F.lit("2026-01-01").cast("date"),
            F.col("id").cast("int"),
        ).alias("month")
    )
    .withColumn("month_end", F.last_day("month"))
)

calendar.show()

# ---------------------------------------------------------
# 2. Create one row per active unit per reporting month.
# ---------------------------------------------------------
active_units = (
    spark_datasets["units"]
    .filter(F.col("active") == 1)
    .select("unit_id", "property_id", "floor_area_sqm")
)

unit_months = active_units.crossJoin(calendar)

# ---------------------------------------------------------
# 3. Match leases active on each month's final day.
# ---------------------------------------------------------
u = unit_months.alias("u")
l = spark_datasets["leases"].alias("l")

active_lease_condition = (
    (F.col("u.unit_id") == F.col("l.unit_id"))
    & (F.col("l.start_date") <= F.col("u.month_end"))
    & (
        F.col("l.end_date").isNull()
        | (F.col("l.end_date") >= F.col("u.month_end"))
    )
)

lease_matches = (
    u.join(l, active_lease_condition, "left")
    .select(
        F.col("u.unit_id").alias("unit_id"),
        F.col("u.property_id").alias("property_id"),
        F.col("u.floor_area_sqm").alias("floor_area_sqm"),
        F.col("u.month").alias("month"),
        F.col("l.lease_id").alias("lease_id"),
        F.col("l.monthly_rent_usd").alias("monthly_rent_usd"),
    )
)

# ---------------------------------------------------------
# 4. Collapse matches to one row per unit-month.
# ---------------------------------------------------------
unit_month_snapshot = (
    lease_matches
    .groupBy("unit_id", "property_id", "month", "floor_area_sqm")
    .agg(
        F.count("lease_id").alias("active_lease_count"),
        F.sum("monthly_rent_usd").alias("matched_rent_usd"),
    )
)

# Stop rather than publish ambiguous rent for conflicting active leases.
lease_conflicts = unit_month_snapshot.filter(
    F.col("active_lease_count") > 1
)

if lease_conflicts.limit(1).count() > 0:
    lease_conflicts.show(20, truncate=False)
    raise ValueError(
        "Overlapping month-end leases detected. "
        "Resolve these conflicts before calculating KPIs."
    )

unit_month_snapshot = (
    unit_month_snapshot
    .withColumn(
        "occupied",
        F.when(F.col("active_lease_count") == 1, 1).otherwise(0),
    )
    # No matching lease means zero contractual rent for that unit-month.
    .withColumn(
        "earned_monthly_rent_usd",
        F.coalesce(
            F.col("matched_rent_usd"),
            F.lit(0).cast("decimal(18,2)"),
        ),
    )
)

# ---------------------------------------------------------
# 5. Aggregate occupancy and rent by property and month.
# ---------------------------------------------------------
occupancy_rent = (
    unit_month_snapshot
    .groupBy("property_id", "month")
    .agg(
        F.count("*").alias("total_units"),
        F.sum("occupied").alias("occupied_units"),
        F.sum("earned_monthly_rent_usd").alias(
            "earned_monthly_rent_usd"
        ),
        # Area includes all active units, including vacant units.
        F.sum("floor_area_sqm").alias("floor_area_sqm"),
    )
    .withColumn(
        "occupancy_rate",
        F.when(
            F.col("total_units") > 0,
            F.col("occupied_units") / F.col("total_units"),
        ),
    )
)

# Include every property-month, even if a property has no active units.
property_months = (
    spark_datasets["properties"]
    .select("property_id")
    .crossJoin(calendar.select("month"))
)

occupancy_rent = (
    property_months
    .join(occupancy_rent, ["property_id", "month"], "left")
    .fillna({
        "total_units": 0,
        "occupied_units": 0,
        "earned_monthly_rent_usd": 0,
        "floor_area_sqm": 0,
    })
)

# ---------------------------------------------------------
# 6. Verify output grain and occupancy limits.
# ---------------------------------------------------------
assert occupancy_rent.count() == 72

assert (
    occupancy_rent
    .groupBy("property_id", "month")
    .count()
    .filter(F.col("count") > 1)
    .count()
) == 0

assert occupancy_rent.filter(
    (F.col("occupied_units") > F.col("total_units"))
    | (F.col("occupancy_rate") < 0)
    | (F.col("occupancy_rate") > 1)
).count() == 0

print("PASS — 72 unique property-month rows with valid occupancy")

occupancy_rent.orderBy("property_id", "month").show(
    12, truncate=False
)

+----------+----------+
|     month| month_end|
+----------+----------+
|2026-01-01|2026-01-31|
|2026-02-01|2026-02-28|
|2026-03-01|2026-03-31|
|2026-04-01|2026-04-30|
|2026-05-01|2026-05-31|
|2026-06-01|2026-06-30|
+----------+----------+

PASS — 72 unique property-month rows with valid occupancy
+-----------+----------+-----------+--------------+-----------------------+--------------+--------------+
|property_id|month     |total_units|occupied_units|earned_monthly_rent_usd|floor_area_sqm|occupancy_rate|
+-----------+----------+-----------+--------------+-----------------------+--------------+--------------+
|P001       |2026-01-01|20         |17            |20550.00               |2040.0        |0.85          |
|P001       |2026-02-01|20         |17            |20550.00               |2040.0        |0.85          |
|P001       |2026-03-01|20         |17            |20550.00               |2040.0        |0.85          |
|P001       |2026-04-01|20         |17            |20550.00      

In [21]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# 1. Attach each work order to its property.
# ---------------------------------------------------------
unit_properties = spark_datasets["units"].select(
    "unit_id", "property_id"
)

maintenance_orders = (
    spark_datasets["work_orders"]
    .join(unit_properties, "unit_id", "inner")
    .withColumn(
        "month",
        F.trunc("opened_date", "month"),
    )
    .filter(
        F.col("month").between("2026-01-01", "2026-06-01")
    )
)

# ---------------------------------------------------------
# 2. Assign response targets by priority.
# ---------------------------------------------------------
maintenance_orders = maintenance_orders.withColumn(
    "sla_target_hours",
    F.when(F.col("priority") == "CRITICAL", 4)
    .when(F.col("priority") == "HIGH", 12)
    .when(F.col("priority") == "MEDIUM", 48)
    .when(F.col("priority") == "LOW", 72),
)

resolved = F.col("status") == "RESOLVED"
open_order = F.col("status") == "OPEN"

# Only resolved orders with a known response duration are eligible.
eligible = (
    resolved
    & F.col("response_hours").isNotNull()
    & F.col("sla_target_hours").isNotNull()
)

compliant = (
    eligible
    & (F.col("response_hours") <= F.col("sla_target_hours"))
)

noncompliant = (
    eligible
    & (F.col("response_hours") > F.col("sla_target_hours"))
)

# ---------------------------------------------------------
# 3. Aggregate maintenance independently of leases.
# ---------------------------------------------------------
maintenance_monthly = (
    maintenance_orders
    .groupBy("property_id", "month")
    .agg(
        F.count("*").alias("total_orders"),

        F.sum(
            F.when(resolved, 1).otherwise(0)
        ).alias("resolved_orders"),

        F.sum(
            F.when(eligible, 1).otherwise(0)
        ).alias("sla_eligible_orders"),

        F.sum(
            F.when(compliant, 1).otherwise(0)
        ).alias("sla_compliant_orders"),

        F.sum(
            F.when(noncompliant, 1).otherwise(0)
        ).alias("sla_noncompliant_orders"),

        F.sum(
            F.when(open_order, 1).otherwise(0)
        ).alias("open_orders"),

        F.sum(
            F.when(
                resolved & F.col("response_hours").isNull(),
                1,
            ).otherwise(0)
        ).alias("missing_response_resolved_orders"),
    )
)

# Include property-months with no maintenance orders.
maintenance_monthly = (
    property_months
    .join(
        maintenance_monthly,
        ["property_id", "month"],
        "left",
    )
    .fillna({
        "total_orders": 0,
        "resolved_orders": 0,
        "sla_eligible_orders": 0,
        "sla_compliant_orders": 0,
        "sla_noncompliant_orders": 0,
        "open_orders": 0,
        "missing_response_resolved_orders": 0,
    })
    .withColumn(
        "sla_compliance_rate",
        F.when(
            F.col("sla_eligible_orders") > 0,
            F.col("sla_compliant_orders")
            / F.col("sla_eligible_orders"),
        ),
    )
)

# Leave the rate NULL when no resolved orders are SLA-eligible.
# NULL means "not calculable", rather than 0% compliance.

# ---------------------------------------------------------
# 4. Verify counts and denominator logic.
# ---------------------------------------------------------
assert maintenance_monthly.count() == 72

assert (
    maintenance_monthly
    .groupBy("property_id", "month")
    .count()
    .filter(F.col("count") != 1)
    .count()
) == 0

# Every eligible order must be compliant or noncompliant.
assert maintenance_monthly.filter(
    F.col("sla_eligible_orders")
    != (
        F.col("sla_compliant_orders")
        + F.col("sla_noncompliant_orders")
    )
).count() == 0

# Every resolved order must be eligible or missing a response.
assert maintenance_monthly.filter(
    F.col("resolved_orders")
    != (
        F.col("sla_eligible_orders")
        + F.col("missing_response_resolved_orders")
    )
).count() == 0

# Every order must be either OPEN or RESOLVED.
assert maintenance_monthly.filter(
    F.col("total_orders")
    != (F.col("resolved_orders") + F.col("open_orders"))
).count() == 0

# Validate both rate ranges and NULL-denominator behaviour.
assert maintenance_monthly.filter(
    (F.col("sla_compliance_rate") < 0)
    | (F.col("sla_compliance_rate") > 1)
    | (
        (F.col("sla_eligible_orders") == 0)
        & F.col("sla_compliance_rate").isNotNull()
    )
    | (
        (F.col("sla_eligible_orders") > 0)
        & F.col("sla_compliance_rate").isNull()
    )
).count() == 0

# Confirm that the property join and reporting window lost no orders.
loaded_order_count = spark_datasets["work_orders"].count()
aggregated_order_count = maintenance_monthly.agg(
    F.sum("total_orders")
).first()[0]

assert aggregated_order_count == loaded_order_count

print("PASS — maintenance counts reconcile with validated work orders")
print("PASS — 72 unique property-month rows with valid SLA rates")

maintenance_monthly.orderBy(
    "property_id", "month"
).show(12, truncate=False)

PASS — maintenance counts reconcile with validated work orders
PASS — 72 unique property-month rows with valid SLA rates
+-----------+----------+------------+---------------+-------------------+--------------------+-----------------------+-----------+--------------------------------+-------------------+
|property_id|month     |total_orders|resolved_orders|sla_eligible_orders|sla_compliant_orders|sla_noncompliant_orders|open_orders|missing_response_resolved_orders|sla_compliance_rate|
+-----------+----------+------------+---------------+-------------------+--------------------+-----------------------+-----------+--------------------------------+-------------------+
|P001       |2026-01-01|16          |14             |14                 |4                   |10                     |2          |0                               |0.2857142857142857 |
|P001       |2026-02-01|20          |18             |18                 |7                   |11                     |2          |0            

In [22]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# 1. Attach valid energy readings to their properties.
# ---------------------------------------------------------
energy_readings = (
    spark_datasets["meter_readings"]
    .join(
        spark_datasets["units"].select(
            "unit_id", "property_id", "active"
        ),
        "unit_id",
        "inner",
    )
    # Match the area denominator: include active units only.
    .filter(F.col("active") == 1)
    .withColumnRenamed("reading_month", "month")
)

# ---------------------------------------------------------
# 2. Aggregate energy independently of leases and work orders.
# ---------------------------------------------------------
energy_monthly = (
    energy_readings
    .groupBy("property_id", "month")
    .agg(
        F.sum("energy_kwh").alias("energy_kwh"),
        F.countDistinct("unit_id").alias("units_with_energy_reading"),
    )
)

# ---------------------------------------------------------
# 3. Join the independently aggregated property-month tables.
# ---------------------------------------------------------
property_month_kpis = (
    occupancy_rent
    .join(
        maintenance_monthly,
        ["property_id", "month"],
        "left",
    )
    .join(
        energy_monthly,
        ["property_id", "month"],
        "left",
    )
    .fillna({"units_with_energy_reading": 0})
    .withColumn(
        "energy_reading_coverage_rate",
        F.when(
            F.col("total_units") > 0,
            F.col("units_with_energy_reading")
            / F.col("total_units"),
        ),
    )
    .withColumn(
        "energy_intensity_kwh_sqm",
        F.when(
            F.col("floor_area_sqm") > 0,
            F.col("energy_kwh") / F.col("floor_area_sqm"),
        ),
    )
)

# Missing energy totals remain NULL, rather than being assumed zero.
# When some readings are missing, intensity uses the available valid
# consumption divided by all active floor area. Coverage flags the gap.

# ---------------------------------------------------------
# 4. Validate the final output grain and energy logic.
# ---------------------------------------------------------
assert property_month_kpis.count() == 72

assert (
    property_month_kpis
    .groupBy("property_id", "month")
    .count()
    .filter(F.col("count") != 1)
    .count()
) == 0

assert property_month_kpis.filter(
    (F.col("energy_kwh") < 0)
    | (F.col("energy_intensity_kwh_sqm") < 0)
    | (F.col("units_with_energy_reading") > F.col("total_units"))
    | (F.col("energy_reading_coverage_rate") < 0)
    | (F.col("energy_reading_coverage_rate") > 1)
    | (
        (F.col("floor_area_sqm") <= 0)
        & F.col("energy_intensity_kwh_sqm").isNotNull()
    )
).count() == 0

# Confirm that joining maintenance and energy did not multiply rent.
rent_before = occupancy_rent.agg(
    F.sum("earned_monthly_rent_usd")
).first()[0]

rent_after = property_month_kpis.agg(
    F.sum("earned_monthly_rent_usd")
).first()[0]

assert rent_before == rent_after

# Reconcile energy totals for EACH month against validated readings.
# Decimal sums allow exact comparison without rounding.
energy_expected = (
    energy_readings
    .groupBy("month")
    .agg(F.sum("energy_kwh").alias("expected_energy_kwh"))
)

energy_actual = (
    property_month_kpis
    .groupBy("month")
    .agg(F.sum("energy_kwh").alias("actual_energy_kwh"))
)

energy_reconciliation = (
    energy_expected
    .join(energy_actual, "month", "full")
    .withColumn(
        "matches",
        F.col("expected_energy_kwh").eqNullSafe(
            F.col("actual_energy_kwh")
        ),
    )
)

assert energy_reconciliation.filter(
    ~F.col("matches")
).count() == 0

print("PASS — 72 unique final property-month KPI rows")
print("PASS — rent unchanged after KPI joins")
print("PASS — monthly energy totals reconcile")

energy_reconciliation.orderBy("month").show(truncate=False)

# Display months with incomplete energy coverage.
print("Property-months with incomplete energy coverage:")

property_month_kpis.filter(
    F.col("units_with_energy_reading") < F.col("total_units")
).select(
    "property_id",
    "month",
    "total_units",
    "units_with_energy_reading",
    "energy_reading_coverage_rate",
).orderBy("property_id", "month").show(truncate=False)

# Preview the combined business metrics.
property_month_kpis.select(
    "property_id",
    "month",
    "occupancy_rate",
    "earned_monthly_rent_usd",
    "sla_compliance_rate",
    "energy_kwh",
    "floor_area_sqm",
    "energy_intensity_kwh_sqm",
).orderBy("property_id", "month").show(12, truncate=False)

PASS — 72 unique final property-month KPI rows
PASS — rent unchanged after KPI joins
PASS — monthly energy totals reconcile
+----------+-------------------+-----------------+-------+
|month     |expected_energy_kwh|actual_energy_kwh|matches|
+----------+-------------------+-----------------+-------+
|2026-01-01|160098.66          |160098.66        |true   |
|2026-02-01|169242.08          |169242.08        |true   |
|2026-03-01|166593.93          |166593.93        |true   |
|2026-04-01|165089.40          |165089.40        |true   |
|2026-05-01|167188.95          |167188.95        |true   |
|2026-06-01|161039.67          |161039.67        |true   |
+----------+-------------------+-----------------+-------+

Property-months with incomplete energy coverage:
+-----------+----------+-----------+-------------------------+----------------------------+
|property_id|month     |total_units|units_with_energy_reading|energy_reading_coverage_rate|
+-----------+----------+-----------+----------------

In [26]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

parquet_path = Path(
    "data/curated/property_month_kpis"
).resolve()

# Collect only the small final aggregate, never the raw fact tables.
assert property_month_kpis.count() == 72
kpi_export = property_month_kpis.toPandas()

# Use consistent partition names such as month=2026-01-01.
kpi_export["month"] = pd.to_datetime(
    kpi_export["month"]
).dt.strftime("%Y-%m-%d")

assert not kpi_export.duplicated(
    ["property_id", "month"]
).any()

# Verify all six expected months before replacing their partitions.
expected_months = {
    f"2026-{month:02d}-01"
    for month in range(1, 7)
}
assert set(kpi_export["month"]) == expected_months

# Preserve decimal monetary and energy values when converting to Arrow.
table = pa.Table.from_pandas(
    kpi_export,
    preserve_index=False,
)

# Replace each output partition rather than append duplicate files.
pq.write_to_dataset(
    table,
    root_path=str(parquet_path),
    partition_cols=["month"],
    existing_data_behavior="delete_matching",
    compression="snappy",
)

# Read the files back using PyArrow, bypassing Hadoop's Windows writer.
reloaded = pq.read_table(str(parquet_path)).to_pandas()

# Partition discovery may return month as a categorical column.
reloaded["month"] = reloaded["month"].astype("string")

assert len(reloaded) == 72
assert not reloaded.duplicated(["property_id", "month"]).any()

partition_counts = reloaded.groupby("month").size()
assert set(partition_counts.index) == expected_months
assert partition_counts.eq(12).all()

# Compare every saved value with the Spark-derived export.
def normalise_for_comparison(df):
    result = df.copy()
    result["month"] = result["month"].astype("string")

    return (
        result[kpi_export.columns]
        .sort_values(["property_id", "month"])
        .reset_index(drop=True)
    )

pd.testing.assert_frame_equal(
    normalise_for_comparison(kpi_export),
    normalise_for_comparison(reloaded),
    check_dtype=False,
)

print("PASS — saved 72 unique property-month rows")
print("PASS — six month partitions with 12 properties each")
print("PASS — saved values match the Spark-derived results")
print(f"Parquet saved to: {parquet_path}")

display(partition_counts.rename("row_count").reset_index())

PASS — saved 72 unique property-month rows
PASS — six month partitions with 12 properties each
PASS — saved values match the Spark-derived results
Parquet saved to: C:\Users\Cedri\OneDrive\Documents\10Alytics - Data Engineering\Final Capstone Project\data\curated\property_month_kpis


,month,row_count
0,2026-01-01,12
1,2026-02-01,12
2,2026-03-01,12
3,2026-04-01,12
4,2026-05-01,12
5,2026-06-01,12


### Install the database driver

In [27]:
%pip install "psycopg[binary]"

   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.7 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.7 MB 2.8 MB/s eta 0:00:02
   ----------------- ---------------------- 1.6/3.7 MB 3.1 MB/s eta 0:00:01
   ---------------------- ----------------- 2.1/3.7 MB 3.2 MB/s eta 0:00:01
   ---------------------------- ----------- 2.6/3.7 MB 3.0 MB/s eta 0:00:01
   ---------------------------------- ----- 3.1/3.7 MB 2.9 MB/s eta 0:00:01
   ---------------------------------- ----- 3.1/3.7 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 2.3 MB/s  0:00:01

   ---------------------------------------- 0/2 [psycopg-binary]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- ------------------- 1/2 [psycopg]
   -------------------- -----------

In [28]:
from getpass import getpass
import psycopg

# Prompt privately rather than storing the password in notebook code.
db_password = getpass("PostgreSQL password: ")

try:
    # The connection closes automatically when this block finishes.
    with psycopg.connect(
        host="localhost",
        port=5432,
        dbname="proptech360",
        user="postgres",
        password=db_password,
        connect_timeout=10,
    ) as connection:

        with connection.cursor() as cursor:
            cursor.execute(
                "SELECT current_database(), current_user;"
            )
            database, username = cursor.fetchone()

            print(f"Connected to: {database}")
            print(f"Database user: {username}")

finally:
    # Remove the password variable after the connection attempt.
    del db_password

Connected to: proptech360
Database user: postgres


In [29]:
from getpass import getpass
from src.load_postgres import load_postgres

# Request the password privately; never print the connection parameters.
connection_parameters = {
    "host": "localhost",
    "port": 5432,
    "dbname": "proptech360",
    "user": "postgres",
    "password": getpass("PostgreSQL password: "),
    "connect_timeout": 10,
}

try:
    first_load = load_postgres(connection_parameters)
    second_load = load_postgres(connection_parameters)

    assert first_load == second_load
    assert second_load["properties"] == 12
    assert second_load["months"] == 6
    assert second_load["facts"] == 72

    print("First load:", first_load)
    print("Second load:", second_load)
    print("PASS — repeated upserts retain 72 fact rows")
    print("PASS — stored base values match curated Parquet")

finally:
    connection_parameters.pop("password", None)

First load: {'properties': 12, 'months': 6, 'facts': 72, 'reconciled_input_rows': 72}
Second load: {'properties': 12, 'months': 6, 'facts': 72, 'reconciled_input_rows': 72}
PASS — repeated upserts retain 72 fact rows
PASS — stored base values match curated Parquet


In [30]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

source = Path("aws/lambda_handler.py")
zip_path = Path("aws/proptech360_lambda.zip")

if not source.is_file():
    raise FileNotFoundError(f"Save the handler first: {source}")

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    # Rename inside the ZIP to match Lambda's configured handler module.
    archive.write(source, arcname="lambda_function.py")

print(f"Upload this file: {zip_path.resolve()}")

Upload this file: C:\Users\Cedri\OneDrive\Documents\10Alytics - Data Engineering\Final Capstone Project\aws\proptech360_lambda.zip


In [1]:
from pathlib import Path
import pyarrow.parquet as pq

# Inspect an actual saved Parquet file.
# Month is stored in folder names, so it won't appear inside this file.
parquet_files = sorted(
    Path("data/curated/property_month_kpis").glob("month=*/*.parquet")
)

if not parquet_files:
    raise FileNotFoundError("No curated Parquet files found.")

print(pq.read_schema(parquet_files[0]))

property_id: large_string
total_units: int64
occupied_units: int64
earned_monthly_rent_usd: decimal128(7, 2)
floor_area_sqm: double
occupancy_rate: double
total_orders: int64
resolved_orders: int64
sla_eligible_orders: int64
sla_compliant_orders: int64
sla_noncompliant_orders: int64
open_orders: int64
missing_response_resolved_orders: int64
sla_compliance_rate: double
energy_kwh: decimal128(7, 2)
units_with_energy_reading: int64
energy_reading_coverage_rate: double
energy_intensity_kwh_sqm: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2668


In [ ]:
from pathlib import Path

# Check the expected submission files.
required_files = [
    "capstone.ipynb",
    "README.md",
    "requirements.txt",
    "src/ingest.py",
    "src/quality.py",
    "src/load_postgres.py",
    "sql/01_schema.sql",
    "sql/03_analytics.sql",
    "aws/lambda_handler.py",
    "aws/iam_policy.json",
    "aws/sample_s3_event.json",
    "aws/athena_ddl.sql",
    "aws/athena_queries.sql",
    "tests/test_ingest.py",
    "tests/test_quality.py",
    "evidence/27_pytest_combined.png",
]

for filename in required_files:
    status = "FOUND" if Path(filename).is_file() else "MISSING"
    print(f"{status}: {filename}")

FOUND: capstone.ipynb
FOUND: README.md
FOUND: requirements.txt
FOUND: src/ingest.py
FOUND: src/quality.py
FOUND: src/load_postgres.py
FOUND: sql/01_schema.sql
FOUND: sql/03_analytics.sql
FOUND: aws/lambda_handler.py
FOUND: aws/iam_policy.json
FOUND: aws/sample_s3_event.json
FOUND: aws/athena_ddl.sql
FOUND: aws/athena_queries.sql
FOUND: tests/test_ingest.py
FOUND: tests/test_quality.py
FOUND: evidence/27_pytest_combined.png


### FINAL REFELCTION

The duplicate records were the main data quality issue I had to address. Leaving them in the data could have affected the unit counts and maintenance results. I also found unknown unit references, a negative repair cost, and a negative energy reading. I quarantined these records instead of guessing replacement values.

Using one row per property per month made the results easier to compare. I calculated occupancy, rent, maintenance, and energy separately before joining them. This prevented duplicate counting and produced 72 reporting rows.

One limitation of my AWS setup is that uploading a raw file does not run the entire pipeline. Lambda checks arrival metadata, while I run the cleaning and Spark calculations locally and upload the Parquet files myself. For production, I would automate these steps, add failure alerts, and improve the handling of missing energy readings.

I really enjoyed working on this project. It was challenging and exciting, and I learnt a lot while solving problems along the way. Getting Spark working, connecting PostgreSQL, and seeing matching results in Athena gave me a real sense of achievement. I pushed myself beyond what I initially thought I could do and felt that I outdid myself.

The experience taught me that getting an output is only part of the work. Checking accuracy, explaining decisions, and proving that results match are just as necessary.

In [6]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

project_dir = Path.cwd()
zip_path = project_dir / "PropTech360_Submission.zip"

# Include the notebook, documentation, code, data, tests, and evidence.
items = [
    "capstone.ipynb",
    "README.md",
    "requirements.txt",
    "src",
    "sql",
    "aws",
    "tests",
    "data",
    "evidence",
]

# Exclude generated caches and the redundant Lambda deployment ZIP.
excluded_dirs = {"__pycache__", ".pytest_cache", ".ipynb_checkpoints"}

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for item in items:
        path = project_dir / item

        if not path.exists():
            raise FileNotFoundError(f"Missing submission item: {item}")

        candidates = [path] if path.is_file() else path.rglob("*")

        for file in candidates:
            if not file.is_file():
                continue

            relative = file.relative_to(project_dir)

            if any(part in excluded_dirs for part in relative.parts):
                continue

            if file.suffix.lower() in {".pyc", ".zip"}:
                continue

            archive.write(file, arcname=relative.as_posix())

# Verify the archive can be read without corruption.
with ZipFile(zip_path) as archive:
    assert archive.testzip() is None
    print(f"Files included: {len(archive.namelist())}")

print(f"Submission ZIP created: {zip_path}")
print(f"Size: {zip_path.stat().st_size / (1024 ** 2):.2f} MB")

Files included: 112
Submission ZIP created: c:\Users\Cedri\OneDrive\Documents\10Alytics - Data Engineering\Final Capstone Project\PropTech360_Submission.zip
Size: 2.34 MB
